# Agricultural Climate Data Analysis
## Helping Farmers Anticipate Demand, Pricing & Climate Risks

This analysis explores 1000+ records of agricultural data across multiple crops and regions to identify:
- Market demand patterns
- Price fluctuations and trends
- Climate risks (drought, flood, frost, heat stress, disease)
- Yield predictions based on weather patterns

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("Agricultural Climate Data Analysis System")
print("="*70)

## Step 1: Generate Comprehensive Agricultural Dataset (1000+ rows)

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate 1000+ rows of agricultural data
print("\nGenerating Agricultural Climate Dataset...")
print("="*70)

# Date range: 3 years of daily data
start_date = datetime(2021, 1, 1)
dates = [start_date + timedelta(days=x) for x in range(1095)]  # 3 years

# Crops and regions
crops = ['Wheat', 'Corn', 'Soybeans', 'Rice', 'Barley']
regions = ['North Valley', 'South Plains', 'East Coast', 'West Ridge', 'Central Basin']

# Generate data
data = []
for date in dates:
    for region in regions:
        for crop in crops:
            # Weather patterns with seasonality
            day_of_year = date.timetuple().tm_yday
            seasonal_temp = 20 + 15 * np.sin(2 * np.pi * day_of_year / 365)
            seasonal_rain = 50 + 30 * np.sin(2 * np.pi * (day_of_year - 90) / 365)
            
            # Add noise
            temperature = seasonal_temp + np.random.normal(0, 5)
            rainfall = max(0, seasonal_rain + np.random.normal(0, 15))
            humidity = 60 + np.random.normal(0, 10)
            soil_moisture = 50 + np.random.normal(0, 15)
            
            # Crop yield (affected by weather)
            base_yield = {'Wheat': 4000, 'Corn': 9000, 'Soybeans': 2500, 'Rice': 6000, 'Barley': 3500}
            weather_factor = (temperature - 20) * 10 + rainfall * 2 + soil_moisture * 5
            yield_kg_per_hectare = base_yield[crop] + weather_factor + np.random.normal(0, 200)
            yield_kg_per_hectare = max(100, yield_kg_per_hectare)
            
            # Market price (inversely related to yield/supply)
            base_price = {'Wheat': 250, 'Corn': 200, 'Soybeans': 400, 'Rice': 350, 'Barley': 220}
            price = base_price[crop] * (1 - yield_kg_per_hectare / 10000) + np.random.normal(0, 20)
            price = max(50, price)
            
            # Demand (seasonal patterns)
            base_demand = {'Wheat': 50000, 'Corn': 80000, 'Soybeans': 30000, 'Rice': 60000, 'Barley': 25000}
            demand = base_demand[crop] * (1 + 0.3 * np.sin(2 * np.pi * day_of_year / 365)) + np.random.normal(0, 5000)
            demand = max(1000, demand)
            
            # Risk indicators
            drought_risk = 1 if rainfall < 30 else (0.5 if rainfall < 50 else 0)
            flood_risk = 1 if rainfall > 150 else (0.5 if rainfall > 100 else 0)
            frost_risk = 1 if temperature < 0 else 0
            heat_stress = 1 if temperature > 35 else (0.5 if temperature > 30 else 0)
            disease_risk = humidity * 0.8 / 100
            
            data.append({
                'Date': date,
                'Region': region,
                'Crop': crop,
                'Temperature_C': temperature,
                'Rainfall_mm': rainfall,
                'Humidity_percent': humidity,
                'Soil_Moisture_percent': soil_moisture,
                'Yield_kg_per_hectare': yield_kg_per_hectare,
                'Market_Price_per_kg': price,
                'Market_Demand_kg': demand,
                'Drought_Risk': drought_risk,
                'Flood_Risk': flood_risk,
                'Frost_Risk': frost_risk,
                'Heat_Stress_Risk': heat_stress,
                'Disease_Risk': disease_risk
            })

df = pd.DataFrame(data)
print(f"\n✓ Dataset created with {len(df)} rows")
print(f"✓ Time period: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"✓ Regions: {df['Region'].nunique()}")
print(f"✓ Crops: {df['Crop'].nunique()}")
print(f"\n{df.head(10)}")

## Step 2: Data Overview & Summary Statistics

In [ ]:
print("\n" + "="*70)
print("DATASET SUMMARY STATISTICS")
print("="*70)
print(f"\nTotal Records: {len(df):,}")
print(f"Date Range: {df['Date'].min()} to {df['Date'].max()}")
print(f"\nUnique Values:")
print(f"  - Crops: {sorted(df['Crop'].unique())}")
print(f"  - Regions: {sorted(df['Region'].unique())}")
print(f"\nStatistical Summary:")
print(df[['Temperature_C', 'Rainfall_mm', 'Humidity_percent', 'Soil_Moisture_percent', 
          'Yield_kg_per_hectare', 'Market_Price_per_kg', 'Market_Demand_kg']].describe().round(2))

## Step 3: Climate Risk Assessment

In [ ]:
print("\n" + "="*70)
print("CLIMATE RISK ANALYSIS")
print("="*70)

# Risk summary
print(f"\nRisk Occurrences (% of records):")
print(f"  - Drought Risk (High): {(df['Drought_Risk'] > 0.5).sum() / len(df) * 100:.1f}%")
print(f"  - Flood Risk (High): {(df['Flood_Risk'] > 0.5).sum() / len(df) * 100:.1f}%")
print(f"  - Frost Risk: {(df['Frost_Risk'] > 0).sum() / len(df) * 100:.1f}%")
print(f"  - Heat Stress: {(df['Heat_Stress_Risk'] > 0.5).sum() / len(df) * 100:.1f}%")
print(f"  - Disease Risk (High): {(df['Disease_Risk'] > 0.6).sum() / len(df) * 100:.1f}%")

# Risk by region
print(f"\nHigh Risk Events by Region:")
risk_by_region = df.groupby('Region').agg({
    'Drought_Risk': lambda x: (x > 0.5).sum(),
    'Flood_Risk': lambda x: (x > 0.5).sum(),
    'Frost_Risk': 'sum',
    'Heat_Stress_Risk': lambda x: (x > 0.5).sum()
})
print(risk_by_region)

## Step 4: Market Demand & Pricing Analysis

In [ ]:
print("\n" + "="*70)
print("MARKET ANALYSIS")
print("="*70)

# Average price and demand by crop
market_summary = df.groupby('Crop').agg({
    'Market_Price_per_kg': ['mean', 'min', 'max', 'std'],
    'Market_Demand_kg': ['mean', 'min', 'max'],
    'Yield_kg_per_hectare': 'mean'
}).round(2)

print("\nMarket Summary by Crop:")
print(market_summary)

# Demand seasonality
df['Month'] = pd.to_datetime(df['Date']).dt.month
seasonality = df.groupby(['Crop', 'Month'])['Market_Demand_kg'].mean().unstack()
print("\nAverage Monthly Demand (kg) by Crop:")
print(seasonality.round(0))

## Step 5: Yield Prediction - Weather Impact Analysis

In [ ]:
print("\n" + "="*70)
print("YIELD PREDICTION & WEATHER CORRELATION")
print("="*70)

# Calculate correlations
print("\nCorrelation with Yield (kg/hectare):")
correlation_cols = ['Temperature_C', 'Rainfall_mm', 'Humidity_percent', 'Soil_Moisture_percent']
correlations = df[correlation_cols + ['Yield_kg_per_hectare']].corr()['Yield_kg_per_hectare'].drop('Yield_kg_per_hectare').sort_values(ascending=False)
print(correlations.round(3))

# Yield by region
print("\nAverage Yield by Region and Crop:")
yield_analysis = df.groupby(['Region', 'Crop'])['Yield_kg_per_hectare'].mean().unstack().round(0)
print(yield_analysis)

## VISUALIZATION 1: Climate Risk Heatmap

In [ ]:
# Create risk heatmap by region and crop
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Climate Risk Assessment by Region & Crop', fontsize=16, fontweight='bold', y=0.995)

risks = [
    ('Drought_Risk', 'Drought Risk'),
    ('Flood_Risk', 'Flood Risk'),
    ('Heat_Stress_Risk', 'Heat Stress Risk'),
    ('Disease_Risk', 'Disease Risk')
]

for idx, (risk_col, risk_label) in enumerate(risks):
    ax = axes[idx // 2, idx % 2]
    risk_matrix = df.groupby(['Region', 'Crop'])[risk_col].mean().unstack()
    sns.heatmap(risk_matrix, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=ax, cbar_kws={'label': 'Risk Level'})
    ax.set_title(f'{risk_label} by Region & Crop', fontsize=12, fontweight='bold')
    ax.set_xlabel('Crop')
    ax.set_ylabel('Region')

plt.tight_layout()
plt.savefig('climate_risk_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Risk heatmap saved!")

## VISUALIZATION 2: Price & Demand Trends Over Time

In [ ]:
# Monthly average price and demand for top crops
df['YearMonth'] = pd.to_datetime(df['Date']).dt.to_period('M')

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Market Trends: Price & Demand Over 3 Years', fontsize=14, fontweight='bold')

# Price trends
ax1 = axes[0]
for crop in crops:
    crop_data = df[df['Crop'] == crop].groupby('YearMonth')['Market_Price_per_kg'].mean()
    ax1.plot(crop_data.index.astype(str), crop_data.values, marker='o', label=crop, linewidth=2, markersize=4)
ax1.set_title('Average Market Price Trends by Crop', fontsize=12, fontweight='bold')
ax1.set_xlabel('Month')
ax1.set_ylabel('Price ($/kg)')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Demand trends
ax2 = axes[1]
for crop in crops:
    crop_data = df[df['Crop'] == crop].groupby('YearMonth')['Market_Demand_kg'].mean()
    ax2.plot(crop_data.index.astype(str), crop_data.values, marker='s', label=crop, linewidth=2, markersize=4)
ax2.set_title('Average Market Demand Trends by Crop', fontsize=12, fontweight='bold')
ax2.set_xlabel('Month')
ax2.set_ylabel('Demand (kg)')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('market_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Market trends graph saved!")

## VISUALIZATION 3: Yield vs Weather Correlation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Yield Prediction: Weather Impact Analysis', fontsize=14, fontweight='bold')

weather_factors = [
    ('Temperature_C', 'Temperature (°C)'),
    ('Rainfall_mm', 'Rainfall (mm)'),
    ('Humidity_percent', 'Humidity (%)'),
    ('Soil_Moisture_percent', 'Soil Moisture (%)')
]

for idx, (factor, label) in enumerate(weather_factors):
    ax = axes[idx // 2, idx % 2]
    ax.scatter(df[factor], df['Yield_kg_per_hectare'], alpha=0.3, s=20)
    
    # Add trend line
    z = np.polyfit(df[factor], df['Yield_kg_per_hectare'], 1)
    p = np.poly1d(z)
    x_sorted = np.sort(df[factor])
    ax.plot(x_sorted, p(x_sorted), "r-", linewidth=2, label='Trend')
    
    corr = df[factor].corr(df['Yield_kg_per_hectare'])
    ax.set_title(f'Yield vs {label}\nCorrelation: {corr:.3f}', fontsize=11, fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Yield (kg/hectare)')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig('yield_weather_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Yield correlation graph saved!")

## VISUALIZATION 4: Regional Climate Risk Comparison

In [ ]:
# Regional climate comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Regional Climate & Weather Patterns Comparison', fontsize=14, fontweight='bold')

metrics = [
    ('Temperature_C', 'Temperature (°C)'),
    ('Rainfall_mm', 'Rainfall (mm)'),
    ('Humidity_percent', 'Humidity (%)'),
    ('Soil_Moisture_percent', 'Soil Moisture (%)'),
    ('Yield_kg_per_hectare', 'Average Yield'),
    ('Market_Price_per_kg', 'Market Price')
]

for idx, (metric, label) in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    region_data = df.groupby('Region')[metric].mean().sort_values(ascending=False)
    bars = ax.bar(region_data.index, region_data.values, color=sns.color_palette('husl', len(region_data)))
    ax.set_title(f'{label} by Region', fontsize=11, fontweight='bold')
    ax.set_ylabel(label)
    ax.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('regional_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Regional comparison graph saved!")

## VISUALIZATION 5: Crop Performance & Profitability Analysis

In [ ]:
# Calculate profitability metric
df['Profitability'] = (df['Yield_kg_per_hectare'] * df['Market_Price_per_kg']) / 10000

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Crop Performance & Profitability Analysis', fontsize=14, fontweight='bold')

# Box plot of yield by crop
ax1 = axes[0]
crop_yields = [df[df['Crop'] == crop]['Yield_kg_per_hectare'].values for crop in crops]
bp = ax1.boxplot(crop_yields, labels=crops, patch_artist=True)
for patch, color in zip(bp['boxes'], sns.color_palette('husl', len(crops))):
    patch.set_facecolor(color)
ax1.set_title('Yield Distribution by Crop', fontsize=12, fontweight='bold')
ax1.set_ylabel('Yield (kg/hectare)')
ax1.grid(True, alpha=0.3, axis='y')

# Profitability by crop
ax2 = axes[1]
prof_by_crop = df.groupby('Crop')['Profitability'].agg(['mean', 'std'])
bars = ax2.bar(prof_by_crop.index, prof_by_crop['mean'], yerr=prof_by_crop['std'], 
                capsize=5, color=sns.color_palette('husl', len(prof_by_crop)), alpha=0.7)
ax2.set_title('Average Profitability by Crop (with Std Dev)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Profitability Index')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('crop_profitability.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Crop profitability graph saved!")

## Key Insights & Recommendations for Farmers

In [ ]:
print("\n" + "="*70)
print("KEY INSIGHTS & RECOMMENDATIONS FOR FARMERS")
print("="*70)

# 1. Best performing crop
best_crop = df.groupby('Crop')['Yield_kg_per_hectare'].mean().idxmax()
best_crop_yield = df.groupby('Crop')['Yield_kg_per_hectare'].mean().max()
print(f"\n1. HIGHEST YIELD CROP:")
print(f"   → {best_crop}: {best_crop_yield:.0f} kg/hectare average")
print(f"   ✓ Best for maximizing production volume")

# 2. Most profitable crop
best_profit_crop = df.groupby('Crop')['Profitability'].mean().idxmax()
best_profit = df.groupby('Crop')['Profitability'].mean().max()
print(f"\n2. MOST PROFITABLE CROP:")
print(f"   → {best_profit_crop}: {best_profit:.2f} profitability index")
print(f"   ✓ Best for revenue maximization")

# 3. Most stable (lowest price volatility)
price_std = df.groupby('Crop')['Market_Price_per_kg'].std()
most_stable = price_std.idxmin()
print(f"\n3. MOST PRICE STABLE CROP:")
print(f"   → {most_stable}: ±${price_std[most_stable]:.2f} price volatility")
print(f"   ✓ Best for risk-averse farmers")

# 4. Best region for farming
yield_by_region = df.groupby('Region')['Yield_kg_per_hectare'].mean()
best_region = yield_by_region.idxmax()
print(f"\n4. BEST FARMING REGION:")
print(f"   → {best_region}: {yield_by_region[best_region]:.0f} kg/hectare average")
print(f"   ✓ Optimal climate conditions for high yields")

# 5. Climate risk hotspots
df['Total_Risk'] = df['Drought_Risk'] + df['Flood_Risk'] + df['Frost_Risk'] + df['Heat_Stress_Risk'] + df['Disease_Risk']
highest_risk_region = df.groupby('Region')['Total_Risk'].mean().idxmax()
risk_level = df.groupby('Region')['Total_Risk'].mean().max()
print(f"\n5. HIGHEST CLIMATE RISK REGION:")
print(f"   → {highest_risk_region}: Risk Level {risk_level:.2f}")
print(f"   ✓ Requires additional climate adaptation strategies")

# 6. Seasonal demand patterns
peak_month = df.groupby('Month')['Market_Demand_kg'].mean().idxmax()
print(f"\n6. PEAK DEMAND SEASON:")
print(f"   → Month {peak_month} (typically {['', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December'][peak_month]})")
print(f"   ✓ Plan planting for peak season harvests")

# 7. Price vs Yield inverse relationship
corr_price_yield = df['Market_Price_per_kg'].corr(df['Yield_kg_per_hectare'])
print(f"\n7. PRICE-YIELD RELATIONSHIP:")
print(f"   → Correlation: {corr_price_yield:.3f} (inverse relationship)")
if abs(corr_price_yield) < 0.3:
    print(f"   ✓ Weak relationship - both volume AND price matter for revenue")
else:
    print(f"   ✓ Strong inverse - high yields lead to lower prices")

print("\n" + "="*70)
print("✓ Analysis Complete! Check generated graphs for visual patterns.")
print("="*70)